# M5 GNN Product Profiling

This notebook profiles the 16 M5 products used in `notebooks/29_graph_neural_network.ipynb`.

Goal:
- compute descriptive statistics for each product
- understand demand sparsity and variability before the GNN benchmark
- export a clean profiling table for later comparison notebooks

## Output Table

The final table includes:
- product identifiers and metadata
- mean sales
- standard deviation
- zero rate
- non-zero days
- max sales
- coefficient of variation
- optional intermittent-demand descriptors: `ADI` and `CV2`

The product selection is reproduced with the same configuration as `notebook29`:
- `NUM_PRODUCTS = 16`
- `STATE_ID = 'CA'`
- `SEED = 42`
- `MIN_NONZERO_DAYS = 28`
- `MAX_DAYS = 365`

In [3]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'gnn_model_comparison':
    REPO_ROOT = REPO_ROOT.parent.parent
elif REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from src.data_loaders.load_m5_panel import load_m5_panel_subset

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

In [4]:
DATA_PATH = REPO_ROOT / 'data' / 'raw' / 'm5'
EXPORT_PATH = REPO_ROOT / 'notebooks' / 'gnn_model_comparison' / 'exports' / 'm5_gnn_product_profile.csv'

NUM_PRODUCTS = 16
STATE_ID = 'CA'
STORE_ID = None
CAT_ID = None
DEPT_ID = None
SEED = 42
MIN_NONZERO_DAYS = 28
MAX_DAYS = 365

In [5]:
panel = load_m5_panel_subset(
    base_path=str(DATA_PATH),
    num_products=NUM_PRODUCTS,
    seed=SEED,
    state_id=STATE_ID,
    store_id=STORE_ID,
    cat_id=CAT_ID,
    dept_id=DEPT_ID,
    min_nonzero_days=MIN_NONZERO_DAYS,
    max_days=MAX_DAYS,
)

sales = np.asarray(panel['sales'], dtype=float)
metadata = panel['metadata'].copy().reset_index(drop=True)
dates = pd.DatetimeIndex(panel['dates'])

print(f'sales shape: {sales.shape}')
print(f'date range: {dates.min().date()} -> {dates.max().date()}')
metadata[['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']]

sales shape: (16, 365)
date range: 2015-04-26 -> 2016-04-24


,id,item_id,dept_id,cat_id,store_id,state_id
0,FOODS_2_397_CA_2_validation,FOODS_2_397,FOODS_2,FOODS,CA_2,CA
1,FOODS_2_044_CA_3_validation,FOODS_2_044,FOODS_2,FOODS,CA_3,CA
2,FOODS_3_228_CA_1_validation,FOODS_3_228,FOODS_3,FOODS,CA_1,CA
3,FOODS_3_070_CA_2_validation,FOODS_3_070,FOODS_3,FOODS,CA_2,CA
4,FOODS_3_170_CA_3_validation,FOODS_3_170,FOODS_3,FOODS,CA_3,CA
5,FOODS_3_420_CA_3_validation,FOODS_3_420,FOODS_3,FOODS,CA_3,CA
6,FOODS_3_641_CA_3_validation,FOODS_3_641,FOODS_3,FOODS,CA_3,CA
7,FOODS_3_524_CA_4_validation,FOODS_3_524,FOODS_3,FOODS,CA_4,CA
8,HOBBIES_1_323_CA_3_validation,HOBBIES_1_323,HOBBIES_1,HOBBIES,CA_3,CA
9,HOBBIES_1_133_CA_4_validation,HOBBIES_1_133,HOBBIES_1,HOBBIES,CA_4,CA


In [6]:
def profile_series(series: np.ndarray) -> dict:
    y = np.asarray(series, dtype=float)
    nonzero = y[y > 0]
    mean_sales = float(np.mean(y))
    std_sales = float(np.std(y, ddof=0))
    zero_rate = float(np.mean(y == 0))
    nonzero_days = int(np.sum(y > 0))
    max_sales = float(np.max(y))
    cv = float(std_sales / mean_sales) if mean_sales > 0 else np.nan
    adi = float(len(y) / nonzero_days) if nonzero_days > 0 else np.nan
    nonzero_mean = float(np.mean(nonzero)) if nonzero_days > 0 else np.nan
    nonzero_std = float(np.std(nonzero, ddof=0)) if nonzero_days > 0 else np.nan
    cv2 = float((nonzero_std / nonzero_mean) ** 2) if nonzero_days > 0 and nonzero_mean > 0 else np.nan
    return {
        'mean_sales': mean_sales,
        'std_sales': std_sales,
        'zero_rate': zero_rate,
        'nonzero_days': nonzero_days,
        'max_sales': max_sales,
        'cv': cv,
        'ADI': adi,
        'CV2': cv2,
    }

In [7]:
profile_rows = []
for idx, meta in metadata.iterrows():
    row = meta.to_dict()
    row.update(profile_series(sales[idx]))
    profile_rows.append(row)

profile_df = pd.DataFrame(profile_rows)
profile_df['demand_regime_hint'] = np.select(
    [
        (profile_df['ADI'] >= 1.32) & (profile_df['CV2'] < 0.49),
        (profile_df['ADI'] >= 1.32) & (profile_df['CV2'] >= 0.49),
        (profile_df['ADI'] < 1.32) & (profile_df['CV2'] >= 0.49),
    ],
    ['intermittent', 'lumpy', 'erratic'],
    default='smooth',
)

profile_df = profile_df[
    [
        'id',
        'item_id',
        'dept_id',
        'cat_id',
        'store_id',
        'state_id',
        'mean_sales',
        'std_sales',
        'zero_rate',
        'nonzero_days',
        'max_sales',
        'cv',
        'ADI',
        'CV2',
        'demand_regime_hint',
    ]
]

profile_df = profile_df.sort_values(['cat_id', 'dept_id', 'store_id', 'item_id']).reset_index(drop=True)
profile_df

,id,item_id,dept_id,cat_id,store_id,state_id,mean_sales,std_sales,zero_rate,nonzero_days,max_sales,cv,ADI,CV2,demand_regime_hint
0,FOODS_2_397_CA_2_validation,FOODS_2_397,FOODS_2,FOODS,CA_2,CA,0.4082,0.7731,0.7315,98,4.0000,1.8939,3.7245,0.2316,intermittent
1,FOODS_2_044_CA_3_validation,FOODS_2_044,FOODS_2,FOODS,CA_3,CA,0.7315,1.1677,0.6082,143,6.0000,1.5963,2.5524,0.3901,intermittent
2,FOODS_3_228_CA_1_validation,FOODS_3_228,FOODS_3,FOODS,CA_1,CA,7.0082,3.4960,0.0110,361,20.0000,0.4988,1.0111,0.2352,smooth
3,FOODS_3_070_CA_2_validation,FOODS_3_070,FOODS_3,FOODS,CA_2,CA,3.5096,7.6619,0.4301,208,76.0000,2.1831,1.7548,2.2859,lumpy
4,FOODS_3_170_CA_3_validation,FOODS_3_170,FOODS_3,FOODS,CA_3,CA,1.4027,1.7594,0.4000,219,12.0000,1.2542,1.6667,0.5439,lumpy
5,FOODS_3_420_CA_3_validation,FOODS_3_420,FOODS_3,FOODS,CA_3,CA,0.5918,0.8730,0.5918,149,5.0000,1.4752,2.4497,0.2966,intermittent
6,FOODS_3_641_CA_3_validation,FOODS_3_641,FOODS_3,FOODS,CA_3,CA,1.0384,1.3902,0.4658,195,9.0000,1.3389,1.8718,0.4919,lumpy
7,FOODS_3_524_CA_4_validation,FOODS_3_524,FOODS_3,FOODS,CA_4,CA,2.0356,3.0532,0.4932,185,17.0000,1.4999,1.9730,0.6471,lumpy
8,HOBBIES_1_323_CA_3_validation,HOBBIES_1_323,HOBBIES_1,HOBBIES,CA_3,CA,1.4493,1.3183,0.2849,261,7.0000,0.9096,1.3985,0.3067,intermittent
9,HOBBIES_1_133_CA_4_validation,HOBBIES_1_133,HOBBIES_1,HOBBIES,CA_4,CA,0.0548,0.2393,0.9479,19,2.0000,4.3675,19.2105,0.0450,intermittent


In [8]:
summary_by_category = (
    profile_df.groupby('cat_id', as_index=False)[['mean_sales', 'std_sales', 'zero_rate', 'cv', 'ADI', 'CV2']]
    .mean()
    .sort_values('zero_rate', ascending=False)
)

summary_by_category

,cat_id,mean_sales,std_sales,zero_rate,cv,ADI,CV2
1,HOBBIES,0.4507,0.5871,0.7445,2.8200,10.0238,0.1366
2,HOUSEHOLD,0.3712,0.6717,0.7164,1.8447,3.6407,0.2206
0,FOODS,2.0908,2.5218,0.4664,1.4676,2.1255,0.6403


In [9]:
profile_df.sort_values(['zero_rate', 'cv'], ascending=[False, False])[
    ['id', 'cat_id', 'mean_sales', 'std_sales', 'zero_rate', 'nonzero_days', 'max_sales', 'cv', 'ADI', 'CV2', 'demand_regime_hint']
]

,id,cat_id,mean_sales,std_sales,zero_rate,nonzero_days,max_sales,cv,ADI,CV2,demand_regime_hint
9,HOBBIES_1_133_CA_4_validation,HOBBIES,0.0548,0.2393,0.9479,19,2.0000,4.3675,19.2105,0.0450,intermittent
10,HOBBIES_1_284_CA_4_validation,HOBBIES,0.0767,0.2860,0.9288,26,2.0000,3.7280,14.0385,0.0612,intermittent
11,HOBBIES_2_019_CA_4_validation,HOBBIES,0.2219,0.5048,0.8164,67,3.0000,2.2749,5.4478,0.1335,intermittent
15,HOUSEHOLD_2_221_CA_4_validation,HOUSEHOLD,0.2795,0.6485,0.7945,75,5.0000,2.3206,4.8667,0.3120,intermittent
0,FOODS_2_397_CA_2_validation,FOODS,0.4082,0.7731,0.7315,98,4.0000,1.8939,3.7245,0.2316,intermittent
14,HOUSEHOLD_2_051_CA_1_validation,HOUSEHOLD,0.4411,0.7903,0.6959,111,4.0000,1.7917,3.2883,0.2804,intermittent
13,HOUSEHOLD_1_531_CA_1_validation,HOUSEHOLD,0.3699,0.6128,0.6959,111,3.0000,1.6568,3.2883,0.1389,intermittent
12,HOUSEHOLD_1_492_CA_1_validation,HOUSEHOLD,0.3945,0.6350,0.6795,117,3.0000,1.6096,3.1197,0.1510,intermittent
1,FOODS_2_044_CA_3_validation,FOODS,0.7315,1.1677,0.6082,143,6.0000,1.5963,2.5524,0.3901,intermittent
5,FOODS_3_420_CA_3_validation,FOODS,0.5918,0.8730,0.5918,149,5.0000,1.4752,2.4497,0.2966,intermittent


In [10]:
EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
profile_df.to_csv(EXPORT_PATH, index=False)
print(f'Saved profiling table to: {EXPORT_PATH}')

Saved profiling table to: c:\Users\braya\Documents\Research\aurex-demand-forecasting-main\notebooks\gnn_model_comparison\exports\m5_gnn_product_profile.csv
